In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from google.colab import userdata
import hashlib, json, os, pathlib, shutil, subprocess, sys, uuid, zipfile
revision = userdata.get('CEG_WM_CONTRASTIVE_LF_REVISION')
if not revision or len(revision) != 40: raise RuntimeError('exact revision secret is unavailable')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['CEG_WM_ROOT_KEY'] = userdata.get('CEG_WM_ROOT_KEY')
os.environ['CEG_WM_CACHE_ROOT'] = '/content/drive/MyDrive/CEG-WM/cache'
os.environ['CEG_WM_PERSISTENT_ROOT'] = '/content/drive/MyDrive/CEG-WM/models'
handoff_root = pathlib.Path('/content/drive/MyDrive/CEG-WM/colab_handoffs/contrastive_lf_branch_attribution_candidate_selection') / revision
handoff = json.loads((handoff_root / 'contrastive_lf_branch_attribution_handoff.json').read_text())
if handoff['repository_revision'] != revision: raise RuntimeError('handoff revision mismatch')
archive = handoff_root / handoff['package_filename']
if hashlib.sha256(archive.read_bytes()).hexdigest() != handoff['package_sha256']: raise RuntimeError('package archive digest mismatch')
extract_root = pathlib.Path('/content') / ('ceg-wm-contrastive-lf-' + uuid.uuid4().hex)
extract_root.mkdir(mode=0o700)
with zipfile.ZipFile(archive) as source: source.extractall(extract_root)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(extract_root / 'requirements_semantic_texture_operational_preflight.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'Pillow==12.3.0'], check=True)
run_id = 'contrastive-lf-branch-attribution-' + uuid.uuid4().hex
output_root = pathlib.Path('/content/drive/MyDrive/CEG-WM/contrastive_lf_branch_attribution_validation/candidate_selection') / run_id


In [ ]:
command = [sys.executable, str(extract_root / 'scripts/experiment_execution/contrastive_lf_branch_attribution_bootstrap.py'), '--expected-revision', revision, '--expected-package-identity', handoff['package_identity'], '--expected-embedded-manifest-sha256', handoff['embedded_manifest_sha256'], '--run-id', run_id, '--output-root', str(output_root)]
completed = subprocess.run(command, check=False, text=True, capture_output=True)
if not output_root.is_dir(): raise RuntimeError('governed delivery directory was not created')
receipt = json.loads((output_root / 'contrastive_lf_execution_receipt.json').read_text())
result = json.loads((output_root / receipt['result_filename']).read_text())
sums = (output_root / 'SHA256SUMS').read_text().splitlines()
for row in sums:
    digest, filename = row.split('  ', 1)
    if hashlib.sha256((output_root / filename).read_bytes()).hexdigest() != digest: raise RuntimeError('delivery checksum mismatch')
if completed.returncode not in (0, 2): raise RuntimeError('bootstrap returned an unsupported code after delivery')
if completed.returncode == 2 and result['result_classification'] not in ('scientific_failure', 'insufficient_evidence', 'operational_failure'): raise RuntimeError('bounded stop classification is invalid')
print(json.dumps({'returncode': completed.returncode, 'result_classification': result['result_classification'], 'run_id': run_id, 'candidate_selection_passed': result['candidate_selection_passed']}, sort_keys=True))
